# 04 — Validación externa en CSAW-CC

En EMBED (notebook 03), la topología de una ventana de ~3.5 cm en el sitio
donde aparecerá el cáncer, medida en el **cribado anterior leído como
normal**, separaba casos de controles con un efecto modesto (AUC ≈ 0.57).
Era local y no lo explicaban edad, densidad ni BI-RADS B, pero se encontró
tras varias variantes de análisis. Para afirmarlo hacía falta repetirlo en
otra cohorte **con el análisis fijado de antemano**.

Este notebook es esa validación, en **CSAW-CC** (Karolinska, Suecia):

- **El plan** (`docs/plan_validacion_csaw.md`) quedó guardado en un commit
  antes de descargar ninguna imagen de CSAW-CC.
- **El script del análisis** (`scripts/csaw_lesion_site.py`) también, antes
  de tener las imágenes.
- **Lo que no está en el plan** se marca como **exploratorio**.

Cita y condiciones: CSAW-CC, Karolinska Institutet (F. Strand et al.). En
una publicación hay que invitar como coautores a los curadores del dataset
(criterios ICMJE).

## Requisitos previos

1. Acceso a CSAW-CC y credenciales en `~/.aws/credentials`, perfil
   `[csaw-cc]` (nunca en el repo).
2. Metadatos e imágenes: `python scripts/download_csaw.py` (≈ 36 GB en
   `data/csaw/`, fuera de git).


## 0. Imports

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "scripts"))

from src.preprocessing import read_dicom_gray
from src.local_topology import window_around
import download_csaw as dc
import csaw_lesion_site as cs
import embed_lesion_site as ls

CSAW_DIR = ROOT / "data" / "csaw"


## 1. El dataset

Una fila por imagen. Fechas anonimizadas: solo se conoce el **orden** de las
exploraciones de cada mujer (sufijo `_N` del fichero) y `rad_timing` (1 =
cáncer detectado en cribado, 2 = de intervalo, 3 = previas). La edad viene
en dos bandas. Todas las imágenes son Hologic.

In [ ]:
meta = dc.load_metadata(CSAW_DIR / dc.METADATA)
ex = dc.exam_table(meta)
pac = ex.groupby("anon_patientid").agg(case=("case", "first"), n=("seq", "size"))
print(f"{len(pac)} mujeres ({(pac.case == 1).sum()} con cáncer), {len(ex)} exploraciones, {len(meta)} imágenes")
print("exploraciones de cáncer por rad_timing:", ex[ex.case == 1].timing.value_counts().sort_index().to_dict())


## 2. La cohorte del plan

- **Casos:** cáncer detectado en cribado (t), con el cribado inmediatamente
  anterior (t-1) leído como normal; vistas MLO y CC del lado del cáncer que
  tengan el sitio anotado.
- **Controles:** 3 por vista, sin cáncer, con dos cribados normales
  consecutivos, misma banda de edad, misma vista y el número de orden más
  parecido.
- **Sitio en t-1:** la marca del radiólogo en t-1 si existe; si no, la de t
  trasladada con la misma posición relativa en la caja de la mama.

In [ ]:
manifest = pd.read_csv(CSAW_DIR / "manifest_csaw.csv")
y = (manifest["kind"] == "incidente").astype(int).values
casos = manifest[y == 1]
print(f"{casos['pid'].nunique()} cánceres, {len(casos)} vistas de caso, {(1 - y).sum()} controles")
print(f"vistas con el sitio marcado directamente en t-1: {casos['mask_t0'].sum()}")
print("banda de edad (1 = 40-55, 2 = 55+): casos", casos.age.value_counts().sort_index().to_dict(),
      "| controles", manifest[y == 0].age.value_counts().sort_index().to_dict())


## 3. Qué marcó el radiólogo en el cribado anterior

Según el README de CSAW-CC, en el cribado anterior al diagnóstico el
radiólogo **contorneó los signos de cáncer si los había**, y si no había
nada visible **puso un punto donde después apareció el cáncer**. Un ejemplo
de cada uno, con la ventana de 512 px que usa el análisis:

In [ ]:
def marca(fila):
    m = np.array(Image.open(CSAW_DIR / "masks" / fila.file_t0.replace(".dcm", "_mask.png")))
    return m if m.ndim == 2 else m[..., 0]

con_marca = casos[casos.mask_t0]
npx = {r.set_id: int(np.count_nonzero(marca(r))) for r in con_marca.itertuples()}
punto = next(r for r in con_marca.itertuples() if npx[r.set_id] <= 25)
contorno = next(r for r in con_marca.itertuples() if npx[r.set_id] > 25)

fig, axes = plt.subplots(2, 3, figsize=(11, 7.5))
for i, (titulo, fila) in enumerate([("punto: nada visible", punto), ("contorno: signos visibles", contorno)]):
    img_t0, img_t1 = read_dicom_gray(ROOT / fila.path_t0), read_dicom_gray(ROOT / fila.path_t1)
    ys, xs = np.nonzero(marca(fila))
    r, c = ys.mean(), xs.mean()
    axes[i, 0].imshow(img_t0, cmap="gray"); axes[i, 0].plot(c, r, "o", mfc="none", mec="#E8A15C", ms=18, mew=2)
    axes[i, 0].set_title(f"t-1 ({titulo})")
    axes[i, 1].imshow(window_around(img_t0, r, c, cs.HALF), cmap="gray"); axes[i, 1].set_title("ventana en t-1")
    axes[i, 2].imshow(img_t1, cmap="gray"); axes[i, 2].set_title("t (diagnóstico)")
    for ax in axes[i]:
        ax.axis("off")
plt.tight_layout(); plt.show()


## 4. Análisis principal (fijado en el plan)

Medida idéntica a EMBED: ventana de 512×512 px a resolución completa,
256×256, CLAHE, homología cúbica, imagen de persistencia de 6×6 por
dimensión. Random Forest, validación cruzada de 5 particiones agrupada por
paciente del caso, y **1 000 permutaciones** dentro de cada conjunto caso +
controles.

**Criterio de éxito** (del plan): AUC > 0.5, p < 0.05 y OR de la regresión
logística condicional > 1.

Las features se guardan en caché (`data/csaw/csaw_site_cache.pkl`); las
1 000 permutaciones tardan del orden de media hora.

In [ ]:
feats, y, grupos, sets = cs.compute(manifest, verbose=False)
X_ant = np.stack([f["pi_t0"] for f in feats])
X_pos = np.stack([f["pi_t1"] for f in feats])

auc, p = ls.permutation_auc(X_ant, y, grupos, sets, n=1000)
or_, p_or = cs.conditional_or(X_ant, y, grupos, sets)
print(f"Anticipación (t-1 en el sitio): AUC = {auc:.3f}, p (1000 permutaciones) = {p:.4f}")
print(f"Regresión logística condicional: OR por DE = {or_:.2f}, p = {p_or:.2g}")
print("CRITERIO DEL PLAN:", "CUMPLIDO" if (auc > 0.5 and p < 0.05 and or_ > 1) else "NO CUMPLIDO")


## 5. Análisis secundarios (también fijados en el plan)

1. **Control positivo:** la misma ventana en t, con el cáncer presente.
2. **Control negativo de localización:** la ventana desplazada media mama.
3. Por vista. 4. Por dimensión de homología. 5. Según cómo se localizó el
   sitio en t-1 (marcado por el radiólogo frente a trasladado desde t).

200 permutaciones en cada uno.

In [ ]:
sec = {}
sec["1. control positivo (t)"] = ls.permutation_auc(X_pos, y, grupos, sets, n=200)
f_lejos, y_l, g_l, s_l = cs.compute(manifest, shift=0.5, verbose=False)
sec["2. sitio alejado: anticipación"] = ls.permutation_auc(np.stack([f["pi_t0"] for f in f_lejos]), y_l, g_l, s_l, n=200)
sec["2. sitio alejado: control positivo"] = ls.permutation_auc(np.stack([f["pi_t1"] for f in f_lejos]), y_l, g_l, s_l, n=200)
for vw in ("MLO", "CC"):
    m = (manifest["view"] == vw).values
    sec[f"3. solo {vw}"] = ls.permutation_auc(X_ant[m], y[m], grupos[m], sets[m], n=200)
mitad = X_ant.shape[1] // 2
sec["4. solo H0"] = ls.permutation_auc(X_ant[:, :mitad], y, grupos, sets, n=200)
sec["4. solo H1"] = ls.permutation_auc(X_ant[:, mitad:], y, grupos, sets, n=200)
marcado = manifest.groupby("set_id")["mask_t0"].transform("max").astype(bool).values
sec["5. sitio marcado en t-1"] = ls.permutation_auc(X_ant[marcado], y[marcado], grupos[marcado], sets[marcado], n=200)
sec["5. sitio trasladado desde t"] = ls.permutation_auc(X_ant[~marcado], y[~marcado], grupos[~marcado], sets[~marcado], n=200)
pd.DataFrame({"AUC": {k: round(a, 3) for k, (a, _) in sec.items()}, "p": {k: round(q, 3) for k, (_, q) in sec.items()}})


## 6. Exploratorio: ¿veía algo el radiólogo?

**No previsto en el plan.** Si la señal viniera solo de las vistas con
contorno (signos visibles que no se detectaron), la topología estaría
encontrando cánceres omitidos, no un precursor invisible. Separamos las
vistas por el tipo de marca en t-1.

In [ ]:
tipo = manifest["set_id"].map(
    lambda k: "punto (nada visible)" if 0 < npx.get(k, 0) <= 25
    else ("contorno (signos visibles)" if npx.get(k, 0) > 25 else "trasladado desde t"))
expl = {}
for t in ["punto (nada visible)", "contorno (signos visibles)", "trasladado desde t"]:
    m = (tipo == t).values
    a, q = ls.permutation_auc(X_ant[m], y[m], grupos[m], sets[m], n=200)
    expl[t] = {"AUC": round(a, 3), "p": round(q, 3), "vistas de caso": int(y[m].sum())}
pd.DataFrame(expl).T


**Lectura**: con el sitio marcado como **punto** (el radiólogo no veía
nada) la anticipación sigue siendo significativa. La señal no se reduce a
cánceres visibles que se pasaron por alto. Cuando el sitio se **traslada**
desde t, el efecto desaparece: la alineación por la caja de la mama es
demasiado imprecisa para una ventana de 3.5 cm. Eso explica que en EMBED,
donde siempre había que trasladarlo, el efecto fuera más débil.

## 7. EMBED frente a CSAW-CC

Los resultados de EMBED se leen de `results/` (calculados en el
notebook 03).

In [ ]:
import re

def leer(fichero, patron):
    txt = (ROOT / "results" / fichero).read_text(encoding="utf-8")
    return float(re.search(patron + r".*?AUC \(5-fold CV\) = ([0-9.]+)", txt).group(1))

embed = {
    "anticipación (t-1)": leer("embed_lesion_site_lesion_edad.txt", "anticipacion: t-1"),
    "control positivo (t)": leer("embed_lesion_site_lesion_edad.txt", "control positivo"),
    "sitio alejado": leer("embed_lesion_site_lesion_alejado0.5.txt", "anticipacion: t-1"),
}
csaw = {"anticipación (t-1)": auc, "control positivo (t)": sec["1. control positivo (t)"][0],
        "sitio alejado": sec["2. sitio alejado: anticipación"][0]}

fig, ax = plt.subplots(figsize=(7.5, 3.4))
x = np.arange(len(embed)); w = 0.38
ax.bar(x - w / 2, list(embed.values()), w, color="#5B6472", label="EMBED (descubrimiento)")
ax.bar(x + w / 2, list(csaw.values()), w, color="#1F4E8C", label="CSAW-CC (validación)")
ax.axhline(0.5, color="k", ls="--", lw=0.8)
ax.set_xticks(x); ax.set_xticklabels(list(embed)); ax.set_ylim(0.4, 0.8); ax.set_ylabel("AUC")
ax.legend(); plt.tight_layout(); plt.show()


## Conclusión

**El criterio del plan se cumple**: en una cohorte independiente (otro país,
otro hospital) y con el análisis fijado de antemano, la topología de una
ventana en el sitio donde aparecerá el cáncer, medida en el cribado
anterior leído como normal, separa casos de controles (AUC ≈ 0.61).
Además:

- es **local**: con la ventana desplazada media mama, todo cae al azar;
- aparece en MLO y en CC, y en H0 y en H1;
- persiste donde el radiólogo **no veía nada** (marca de punto, análisis
  exploratorio).

Límites:
- efecto modesto (AUC ≈ 0.6), sin utilidad clínica por sí solo;
- en CSAW-CC no se pudo emparejar por intervalo ni fecha (fechas
  anonimizadas);
- cada dataset es de un solo centro;
- el sitio tiene que conocerse: es una pregunta retrospectiva sobre si el
  precursor era medible, no todavía un detector desplegable.
